# Recording the attack

In [2]:
from kafka import KafkaConsumer, TopicPartition
import json
import os, logging
from datetime import datetime, timedelta
from baskervillehall.model_io import ModelIO


In [3]:
kafka_url = ['kafka9-0.kafka9-headless.default.svc.cluster.local:9093','kafka9-1.kafka9-headless.default.svc.cluster.local:9093','kafka9-2.kafka9-headless.default.svc.cluster.local:9093']
topic = 'BASKERVILLEHALL_3'
datetime_format = '%Y-%m-%d %H:%M:%S'
os.environ['S3_ACCESS'] = 'c490f17bdb784fa08f4d11836ee18e48'
os.environ['S3_SECRET'] = 'e4b65e27b3734d0d96ce6038586ef43f'
os.environ['S3_ENDPOINT'] = 's3.gra.cloud.ovh.net'
os.environ['S3_REGION'] = 'GRA'
s3_connection = {
    's3_access':os.environ['S3_ACCESS'],
    's3_secret':os.environ['S3_SECRET'],
    's3_endpoint': os.environ['S3_ENDPOINT'],
    's3_region':os.environ['S3_REGION']
}
partitions = {
    'zhitomir.info': 1,
    'urban-pushkino.ru': 0,
    'dev.emawpb.org': 0,
    'palestinechronicle.com': 1,
    'equalit.ie': 0,
    'lexota.org': 0,
    'kavkaz-uzel.eu': 0,
    'amp.kavkaz-uzel.eu': 2,
    'indymedia.nl': 0,
    'verafiles.org': 1,
    'anticor-kharkiv.org': 0,
}

In [4]:
logger = logging.getLogger('pca')
logger.addHandler(logging.StreamHandler())
logger.setLevel('DEBUG')

In [8]:
def read_dataset(host, partition, start, stop, datetime_format):
    consumer = KafkaConsumer(
        bootstrap_servers=kafka_url,
        group_id='attack2'
    )
    print(f'Reading from kafka. Host = {host} ... partition = {partition}')
    num = 0
    sessions = []
    ips = []

    consumer.assign([TopicPartition(topic, partition)])
    consumer.seek_to_beginning()
    complete = False
    read = 0
    while not complete:
        raw_messages = consumer.poll(timeout_ms=1000, max_records=5000)

        for topic_partition, messages in raw_messages.items():
            for message in messages:
                if message.value is None :
                    continue
                if message.key is None:
                    continue
                session = json.loads(message.value.decode("utf-8"))
                ts = datetime.strptime(session['start'], datetime_format)

                if read % 10000 == 0:
                    print(ts)
                read += 1

                if message.key.decode("utf-8") != host:
                    continue
                    
                if ts > start and ts < stop:
                    sessions.append(session)
                    ips.append(session['ip']) 
                    num += 1
                    if num % 100 == 0:
                        print(f'{num} sessions read', session['end'], message.timestamp)
                if ts > stop:
                    complete = True
                    break
            
    return sessions, ips

In [1]:
host = 'palestinechronicle.com'
start = datetime(2023, 12, 20, 10, 10, 0, 0)
stop =  datetime(2023, 12, 20, 10, 25, 0, 0)

NameError: name 'datetime' is not defined

In [22]:
sessions, ips = read_dataset(host, partitions[host], start, stop, datetime_format)
print(f'{len(sessions)} sessions loaded')

Reading from kafka. Host = palestinechronicle.com ... partition = 1
2023-12-20 02:44:49
2023-12-20 03:53:04
2023-12-20 04:57:28
2023-12-20 05:58:16
2023-12-20 07:00:03
2023-12-20 07:57:00
2023-12-20 08:31:18
2023-12-20 09:03:57
2023-12-20 10:05:33
100 sessions read 2023-12-20 10:16:32 1703067456175
200 sessions read 2023-12-20 10:16:35 1703067498916
300 sessions read 2023-12-20 10:16:33 1703067571638
400 sessions read 2023-12-20 10:16:33 1703067623415
500 sessions read 2023-12-20 10:16:40 1703067650969
600 sessions read 2023-12-20 10:17:59 1703067716253
700 sessions read 2023-12-20 10:17:33 1703067763547
800 sessions read 2023-12-20 10:18:44 1703067763646
900 sessions read 2023-12-20 10:17:48 1703067763754
1000 sessions read 2023-12-20 10:17:52 1703067763872
1100 sessions read 2023-12-20 10:17:59 1703067793730
1200 sessions read 2023-12-20 10:17:50 1703067793832
1300 sessions read 2023-12-20 10:17:59 1703067793932
1400 sessions read 2023-12-20 10:18:17 1703067805206
1482 sessions loade

In [31]:
sessions_dash = []
for session in sessions:
    if session['session_id'] == '-':
        sessions_dash.append(session)    

In [13]:
model_io = ModelIO(**s3_connection, logger=logger)
model = model_io.load('s3://anton/baslervillehall/models22', host)

In [ ]:
scores = model.score_sessions(sessions_dash)
len(scores[scores < 0]) * 1.0 / len(scores)

In [45]:
#def save_dataset(s3_connection, path, name):
path = 'anton/dataset/'
name = 'attack20Dec.json'

model_io._save_object(json.dumps(sessions_dash), os.path.join(path, name))

In [48]:
loaded_sessions = model_io._load_object(os.path.join(path, name))

In [50]:
loaded_sessions = json.loads(loaded_sessions)

In [51]:
len(loaded_sessions)

1295

In [55]:
model_io._save_object(json.dumps(sessions_dash[0:10], indent=2), os.path.join(path, 'test.json'))

In [81]:
def save_dataset(sessions, s3_connection, path, filename):
    model_io = ModelIO(**s3_connection)
    model_io._save_object(json.dumps(sessions, indent=2), os.path.join(path, filename))

def load_dataset(s3_connection, path, filename):
    model_io = ModelIO(**s3_connection)
    return json.loads(model_io._load_object(os.path.join(path, filename)))

In [82]:
test2_origin = sessions[0:10]
save_dataset(test2_origin, s3_connection, path, 'test2.json')

In [83]:
test2 = load_dataset(s3_connection, path, 'test2.json')

In [60]:
model.score_sessions(test2)

array([-0.1504693 , -0.16756061, -0.37416997, ..., -0.36230897,
       -0.37402955, -0.36903966])

In [84]:
test2 == test2_origin

True

In [85]:
len(test2_origin) 

10

In [86]:
len(test2)


10

In [88]:
for s in sessions_dash:
    print(len(s['requests']))

16
30
2453
1765
1097
1003
453
2073
684
1383
720
1031
1771
2704
2743
4039
2626
2756
3507
1972
1388
3198
1925
2259
1670
2681
1014
799
2346
2151
992
2492
1220
2020
1885
3844
1707
1543
3560
3360
532
5504
1726
1695
831
1977
1042
3353
1140
1653
1453
1559
3988
2469
2481
1136
2535
2997
246
1054
1497
3313
307
1760
4059
723
299
2342
1205
16
1747
2688
729
3008
1536
3050
3613
3547
2357
1666
1521
1596
1224
2782
540
1432
1810
1135
2330
1472
1363
2126
1779
2868
1654
2122
996
2513
1618
1657
1916
1284
2336
575
1443
1679
2074
919
1951
1518
993
3190
3582
1006
394
1736
1035
690
1285
1884
877
367
3658
1416
855
963
623
3524
2946
932
1466
1479
1945
805
2660
1960
1862
1608
1641
2580
591
3118
726
1970
955
2011
2561
940
4747
1868
2227
2043
1440
1867
2079
1707
1084
1038
2282
1073
937
1030
1733
690
1581
892
890
331
1207
28
61
152
310
143
401
199
1394
1452
1397
1074
733
1805
915
1645
576
1943
2152
2089
876
1094
1561
1058
851
1181
139
907
339
467
2272
162
762
177
577
121
65
195
231
155
2089
263
70
118
165
43
228
22